In [0]:
-- ============================================================
-- Script : 05_load_facts
-- Purpose: Idempotent load of fact_order_items with a point-in-time
--          join to dim_customer (assigns the customer version that
--          was active on the order date).
-- Depends: brewflow_staging (stg_order_items, stg_orders),
--          brewflow_warehouse (dim_customer, dim_product, dim_store, dim_channel)
-- Outputs: brewflow_warehouse.fact_order_items
--
-- CHANGELOG (2026-09-06):
-- Fix 1 - revenue was missing from the INSERT column list and stayed
--         NULL for every row. Now populated on insert.
-- Fix 2 - fact_order_items._sk columns (esp. product_sk) could go
--         stale if a dimension table is ever rebuilt with new
--         surrogate keys, because the previous version of this
--         script only ever INSERTed new rows and never revisited
--         existing ones. Added a WHEN MATCHED branch so every run
--         re-derives and (if needed) corrects all _sk columns and
--         revenue on already-loaded rows too, making the load
--         self-healing against future surrogate-key drift.
-- ============================================================

MERGE INTO brewflow_warehouse.fact_order_items AS target
USING (
    SELECT
        oi.order_item_id,
        oi.order_id,
        dc.customer_sk,
        dp.product_sk,
        ds.store_sk,
        dch.channel_sk,
        CAST(date_format(o.order_date, 'yyyyMMdd') AS INT) AS date_sk,
        oi.quantity,
        oi.unit_price,
        oi.quantity * oi.unit_price AS revenue
    FROM brewflow_staging.stg_order_items oi
    JOIN brewflow_staging.stg_orders      o   ON oi.order_id  = o.order_id
    -- Point-in-time customer join: the dim_customer row valid on the order date
    JOIN brewflow_warehouse.dim_customer  dc  ON o.customer_id = dc.customer_id
        AND o.order_date BETWEEN dc.effective_from
                             AND COALESCE(dc.effective_to, DATE'9999-12-31')
    JOIN brewflow_warehouse.dim_product   dp  ON oi.product_id = dp.product_id
    JOIN brewflow_warehouse.dim_store     ds  ON o.store_id    = ds.store_id
    JOIN brewflow_warehouse.dim_channel   dch ON o.channel     = dch.channel
) AS source
ON target.order_item_id = source.order_item_id
WHEN MATCHED AND (
        target.customer_sk <> source.customer_sk
     OR target.product_sk  <> source.product_sk
     OR target.store_sk    <> source.store_sk
     OR target.channel_sk  <> source.channel_sk
     OR target.date_sk     <> source.date_sk
     OR target.revenue     <> source.revenue
     OR target.revenue IS NULL
    ) THEN
    UPDATE SET
        target.customer_sk = source.customer_sk,
        target.product_sk  = source.product_sk,
        target.store_sk    = source.store_sk,
        target.channel_sk  = source.channel_sk,
        target.date_sk     = source.date_sk,
        target.quantity    = source.quantity,
        target.unit_price  = source.unit_price,
        target.revenue     = source.revenue
WHEN NOT MATCHED THEN
    INSERT (order_item_id, order_id, customer_sk, product_sk, store_sk, channel_sk,
            date_sk, quantity, unit_price, revenue)
    VALUES (source.order_item_id, source.order_id, source.customer_sk, source.product_sk,
            source.store_sk, source.channel_sk, source.date_sk, source.quantity, source.unit_price,
            source.revenue);

-- Validieren
SELECT 'dim_customer'     AS tbl, COUNT(*) AS rows FROM brewflow_warehouse.dim_customer
UNION ALL SELECT 'dim_product',      COUNT(*) FROM brewflow_warehouse.dim_product
UNION ALL SELECT 'dim_store',        COUNT(*) FROM brewflow_warehouse.dim_store
UNION ALL SELECT 'dim_channel',      COUNT(*) FROM brewflow_warehouse.dim_channel
UNION ALL SELECT 'fact_order_items', COUNT(*) FROM brewflow_warehouse.fact_order_items;
-- Erwartet: 10, 8, 5, 2, 34 (aktueller Stand nach Szenario A-C)

-- Sanity check: keine Zeile darf mehr NULL-revenue oder einen verwaisten _sk haben
SELECT COUNT(*) AS rows_with_null_revenue
FROM brewflow_warehouse.fact_order_items
WHERE revenue IS NULL;
-- Erwartet: 0

SELECT COUNT(*) AS rows_with_invalid_product_sk
FROM brewflow_warehouse.fact_order_items f
LEFT JOIN brewflow_warehouse.dim_product dp ON f.product_sk = dp.product_sk
WHERE dp.product_sk IS NULL;
-- Erwartet: 0

SELECT dp.category, SUM(f.revenue) AS total_revenue
FROM brewflow_warehouse.fact_order_items f
JOIN brewflow_warehouse.dim_product dp ON f.product_sk = dp.product_sk
GROUP BY dp.category
ORDER BY total_revenue DESC;

SELECT dch.channel, SUM(f.revenue) AS total_revenue
FROM brewflow_warehouse.fact_order_items f
JOIN brewflow_warehouse.dim_channel dch ON f.channel_sk = dch.channel_sk
GROUP BY dch.channel;